In [40]:
from bs4 import BeautifulSoup
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torchtext.vocab import build_vocab_from_iterator
import nltk
from nltk.tokenize import word_tokenize

In [2]:
with open('test.html', 'r') as file:
    html_content = file.read()

# Parse the html content using BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')

# Extract the text from the html content
text = soup.get_text()

# remove extra spaces in between words
text = re.sub(r'\s+', ' ', text)

# remove whitespace at the beginning and end of the document
text = text.strip()

print(text)

Dentists Near Me Dentists Near Me Team Services Testimonials Information Contact Quality Dental Care Your great smile begins with a great dentist Book Online Call or Text (613) 123-4567 Meet the Dentists Dr. Kim Campbell Dr. Kim Campbell is a board certified dentist with a broad range of expertise in general dentistry and a special interest in surgery. He is focused on delivering quality care to patients of all ages and medical histories. Why did you become a dentist? I knew I wanted to become a dentist since my fifth grade science fair. For my experiment, I placed teeth in beakers of soda. I was astonished by the changes that occurred to the teeth in such a short period of time. I value my relationship with each and every one of my patients and hope we develop lifelong bonds. I want you to feel happy and confident about your smile and your health. Dr. Jonathan Wang Dr. Jonathan Wang is able to provide his patients with extensive evidence of their excellence through his philosophy, edu

# Preprocessing

In [21]:
nltk.download('punkt')

# Tokenization (character-level and word-level)
def tokenize_char(doc):
    return list(doc)  # Character-level tokenization

def tokenize_word(doc):
    return word_tokenize(doc)  # Word-level tokenization

char_sequences = [tokenize_char(text)]
word_sequences = [tokenize_word(text)] 

# Building the vocab from iterator
def yield_tokens(data_iter):
    for seq in data_iter:
        yield seq

char_vocab = build_vocab_from_iterator(yield_tokens(char_sequences), specials=["<unk>"])
word_vocab = build_vocab_from_iterator(yield_tokens(word_sequences), specials=["<unk>"])

# Setting default index for unknown tokens
char_vocab.set_default_index(char_vocab["<unk>"])
word_vocab.set_default_index(word_vocab["<unk>"])

# Check vocab
print("Character Vocabulary:", char_vocab.get_stoi())  # String-to-index mapping
print("Word Vocabulary:", word_vocab.get_stoi())  # String-to-index mapping


Character Vocabulary: {'q': 69, 'U': 68, 'R': 67, ')': 64, '(': 63, 'L': 61, '8': 57, '2': 56, '&': 55, 'j': 54, 'Q': 66, '5': 53, ':': 52, '?': 58, 'N': 51, 'H': 50, 'B': 49, '7': 47, '1': 41, '!': 38, 'y': 19, '$': 31, 'E': 36, 'Y': 62, 'k': 30, '6': 46, 'x': 28, 'C': 27, '>': 48, 'o': 6, 'O': 44, '4': 65, 'p': 17, 'F': 43, 'D': 26, 'W': 45, 'A': 35, 'S': 33, '3': 42, '0': 23, 'u': 15, 'd': 10, '-': 40, ',': 21, 't': 3, 'n': 5, 'g': 16, 'm': 14, '.': 22, 'M': 37, 'J': 59, 'c': 12, 'v': 24, 'I': 32, 'l': 11, 'e': 2, 'h': 13, 'P': 34, 'K': 60, 'b': 20, '"': 39, 'f': 18, 's': 8, 'T': 29, 'a': 7, 'w': 25, 'i': 4, ' ': 1, 'r': 9, '<unk>': 0}
Word Vocabulary: {'young': 354, 'when': 349, 'we': 348, 'vary': 343, 'unhappy': 341, 'team': 331, 'surgery': 329, 'soda': 325, 'smiling': 323, 'since': 321, 'short': 320, 'science': 319, 'save': 318, 'results': 316, 'restoration': 315, 'quality': 310, 'providers': 308, 'provide': 307, 'process': 306, 'preventative': 304, 'placed': 301, 'philosophy': 3

[nltk_data] Downloading package punkt to /Users/alfred/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [39]:
# Convert to numerical indices
char_indices = [[char_vocab[char] for char in seq] for seq in char_sequences]
word_indices = [[word_vocab[word] for word in seq] for seq in word_sequences]

# Convert to tensors and pad
char_indices = [torch.tensor(seq) for seq in char_indices]
word_indices = [torch.tensor(seq) for seq in word_indices]

char_padded = pad_sequence(char_indices, batch_first=True, padding_value=0)
word_padded = pad_sequence(word_indices, batch_first=True, padding_value=0)


In [41]:
embedding_dim = 100

# Embedding layers
char_embedding = nn.Embedding(len(char_vocab), embedding_dim)
word_embedding = nn.Embedding(len(word_vocab), embedding_dim)

char_embedded = char_embedding(char_padded)
word_embedded = word_embedding(word_padded)

# Concatenate embeddings
concatenated = torch.cat((char_embedded, word_embedded), dim=1)
print(concatenated.shape)

torch.Size([1, 4373, 100])


In [44]:
class HTMLPhishCNN(nn.Module):
    def __init__(self):
        super(HTMLPhishCNN, self).__init__()
        
        # 32 convolutional filters with 8 different kernel sizes
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels=100, out_channels=32, kernel_size=k) for k in [3, 5, 7, 9, 11, 13, 15, 17]
        ])
        
        # Max Pooling layer
        self.pool = nn.MaxPool1d(kernel_size=2)
        
        # Placeholder for fully connected layer; dynamically set later
        self.fc = None
        
        # Output layer for binary classification
        self.output = nn.Linear(10, 1)
    
    def forward(self, x):
        # Apply Conv1D and ReLU to each layer
        x = [F.relu(conv(x.permute(0, 2, 1))) for conv in self.conv_layers]
        
        # Apply Max Pooling to each output
        x = [self.pool(conv) for conv in x]
        
        # Concatenate the output of all convolution layers along the sequence length
        x = torch.cat(x, dim=2)
        
        # Dynamically calculate the flattened size
        if self.fc is None:
            flatten_size = x.view(x.size(0), -1).size(1)
            self.fc = nn.Linear(flatten_size, 10)
        
        # Flatten the output
        x = x.view(x.size(0), -1)
        
        # Fully connected layer
        x = F.relu(self.fc(x))
        
        # Output layer (sigmoid for binary classification)
        x = torch.sigmoid(self.output(x))
        
        return x

# Example: create a model and pass the input
model = HTMLPhishCNN()

# Example input with shape [1, 4373, 100]
input_tensor = concatenated
output = model(input_tensor)
output # Should output the prediction for binary classification


tensor([[0.5432]], grad_fn=<SigmoidBackward0>)